# Promedio de tiempos reales y atrasos hacia Estación Corregidora

Este notebook complementa `02_RutasQroBus.ipynb`. Parte de los horarios programados del GTFS local, consulta **Google Maps Routes API** para obtener la ETA en transporte público desde las paradas hacia la estación Corregidora, conserva observaciones históricas y calcula una corrección promedio.

El resultado principal es `data/processed/tiempos_corregidos_google.csv`. `02_RutasQroBus.ipynb` lo consume de manera opcional para que el filtro de 30 minutos use el tiempo corregido.


## 1. Alcance y definición de las métricas

Se manejan tres medidas distintas:

1. **Tiempo GTFS programado**: suma de los tiempos entre paradas según `stop_times.txt`.
2. **ETA de Google Transit**: duración estimada al momento de la consulta; puede incluir espera, caminata y transbordos.
3. **Atraso estimado (proxy)**: `max(ETA Google - tiempo GTFS, 0)`. Se promedia por ruta y se suma al cálculo GTFS, como se solicitó.

> **Limitación importante:** Google Routes API no entrega un campo de “atraso del camión” ni permite seleccionar un modelo de tráfico cuando `travelMode="TRANSIT"`. La ETA puede incorporar información actualizada cuando el proveedor de transporte la comparte, pero el diferencial contra GTFS también puede contener espera, caminata o transbordos. Por ello se etiqueta como **proxy**, no como atraso operativo observado. Para atraso vehicular exacto se necesita el feed GTFS-Realtime `TripUpdates` de la agencia.

Documentación oficial consultada:

- [Rutas en transporte público](https://developers.google.com/maps/documentation/routes/transit-route)
- [Matriz de rutas en transporte público](https://developers.google.com/maps/documentation/routes/transit-rm)
- [Límites y facturación](https://developers.google.com/maps/documentation/routes/usage-and-billing)
- [Políticas y atribución de Routes API](https://developers.google.com/maps/documentation/routes/policies)


## 2. Configuración segura

La llave nunca se escribe en el notebook. Se carga desde el archivo `.env` ubicado en la raíz del repositorio. Copia `.env.example` como `.env` y completa:

```dotenv
GOOGLE_MAPS_API_KEY=tu_llave
EJECUTAR_GOOGLE_MAPS=true
UMBRAL_ANALISIS_MIN=30
```

El cargador no sobrescribe variables que ya existan en el sistema. `.env` está excluido por `.gitignore`; solo `.env.example`, que no contiene secretos, puede versionarse. La consulta es facturable y permanece apagada salvo que `EJECUTAR_GOOGLE_MAPS=true`.


In [1]:
from pathlib import Path
from datetime import datetime
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

# Los módulos compartidos viven en scripts/; funciona desde notebooks/ o desde la raíz.
SCRIPTS_DIR = next(
    base / "scripts"
    for base in [Path.cwd(), Path.cwd().parent]
    if (base / "scripts" / "config.py").exists()
)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from config import (
    ACTUALIZAR_RED_OSM,
    CACHE_DIR,
    CORREGIDORA_LAT,
    CORREGIDORA_LON,
    EJECUTAR_GOOGLE_MAPS,
    ENV_CARGADO,
    ENV_PATH,
    GOOGLE_MAPS_API_KEY,
    GTFS_DIR,
    MIN_OBSERVACIONES_HISTORICAS,
    PENALIZACION_TRANSBORDO_MIN,
    PROCESSED_DIR,
    RETENCION_HISTORICO_DIAS,
    UMBRAL_ANALISIS_MIN,
    UMBRAL_CONSULTA_GOOGLE_MIN,
)
from geo import (
    TRAZO_RED_VIAL,
    VIA_FERREA_CORREGIDORA_LATLON,
    distancia_firmada_a_via,
    guardar_trazos,
    haversine_m,
)
from google_routes import consultar_paradas_google
from gtfs_network import (
    calcular_rutas_hacia_destino,
    compactar_rutas,
    filtrar_segmentos_mismo_lado_vias,
    gtfs_time_to_seconds,
    guardar_rutas_base,
)
from street_network import (
    FACTORES_VIA_AUTOBUS,
    RedVial,
    cargar_red_osm,
    recortar_red_a_lado_de_via,
)

# None consulta todos los orígenes alcanzables. Use un entero para una prueba controlada.
MAX_PARADAS = None

print(f"Archivo .env cargado: {ENV_CARGADO} ({ENV_PATH})")
print(f"API key configurada: {bool(GOOGLE_MAPS_API_KEY)}")
print(f"GTFS: {GTFS_DIR}")
print(f"Resultados: {PROCESSED_DIR}")
print(f"Consulta a Google habilitada: {EJECUTAR_GOOGLE_MAPS}")
print(f"Paradas consultadas en Google: GTFS <= {UMBRAL_CONSULTA_GOOGLE_MIN:g} min")
print(f"Penalización por transbordo: {PENALIZACION_TRANSBORDO_MIN:g} min")
print(f"Muestra histórica mínima: {MIN_OBSERVACIONES_HISTORICAS} observaciones")


Archivo .env cargado: True (/Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/.env)
API key configurada: True
GTFS: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/gtfs
Resultados: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed
Consulta a Google habilitada: False
Paradas consultadas en Google: GTFS <= 60 min
Penalización por transbordo: 5 min
Muestra histórica mínima: 5 observaciones


## 3. Lectura y validación de los archivos GTFS

Se leen los cuatro archivos desde `data/gtfs/` y se validan las columnas mínimas antes de calcular. Esta validación falla temprano si cambia el esquema de la fuente.


In [2]:
trips = pd.read_csv(GTFS_DIR / "trips.txt")
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")
stops = pd.read_csv(GTFS_DIR / "stops.txt")
routes = pd.read_csv(GTFS_DIR / "routes.txt")

columnas_requeridas = {
    "trips": {"trip_id", "route_id"},
    "stop_times": {
        "trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"
    },
    "stops": {"stop_id", "stop_name", "stop_lat", "stop_lon"},
    "routes": {"route_id", "route_short_name", "route_long_name"},
}
tablas = {
    "trips": trips,
    "stop_times": stop_times,
    "stops": stops,
    "routes": routes,
}

for nombre, requeridas in columnas_requeridas.items():
    faltantes = requeridas - set(tablas[nombre].columns)
    if faltantes:
        raise ValueError(f"{nombre} no contiene las columnas: {sorted(faltantes)}")

if stops["stop_id"].duplicated().any():
    warnings.warn("Hay stop_id duplicados; se conservará el primer registro.")
    stops = stops.drop_duplicates("stop_id", keep="first").copy()

print(
    f"{len(stops):,} paradas, {len(routes):,} rutas, "
    f"{len(trips):,} viajes y {len(stop_times):,} horarios cargados."
)


2,694 paradas, 217 rutas, 32,536 viajes y 1,326,009 horarios cargados.


## 4. Tiempo programado hacia Corregidora

Los horarios GTFS admiten horas mayores a 24:00. Primero se convierten a segundos sin perder el cambio de día. Después se construye un grafo dirigido de segmentos consecutivos y se aplica Dijkstra sobre estados `(parada, ruta)` para encontrar el menor tiempo programado desde cada parada hacia Corregidora, incluyendo la penalización configurada por transbordo. El resultado completo se guarda en `data/processed/rutas_base_gtfs.csv`; `02_RutasQroBus.ipynb` lo reutiliza sin ejecutar Dijkstra otra vez.

Para evitar que una observación extrema gobierne un segmento, el peso de cada combinación `origen-destino-ruta` es su **mediana** programada.


In [3]:
stop_times_work = stop_times.copy()
stop_times_work["arrival_sec"] = gtfs_time_to_seconds(
    stop_times_work["arrival_time"]
)
stop_times_work["departure_sec"] = gtfs_time_to_seconds(
    stop_times_work["departure_time"]
)
stop_times_work = stop_times_work.merge(
    trips[["trip_id", "route_id"]], on="trip_id", how="left", validate="many_to_one"
)
stop_times_work = stop_times_work.sort_values(
    ["trip_id", "stop_sequence"], kind="stable"
)

stop_times_work["next_stop_id"] = (
    stop_times_work.groupby("trip_id", sort=False)["stop_id"].shift(-1)
)
stop_times_work["next_arrival_sec"] = (
    stop_times_work.groupby("trip_id", sort=False)["arrival_sec"].shift(-1)
)

segmentos = stop_times_work.dropna(
    subset=["next_stop_id", "next_arrival_sec", "route_id"]
).copy()

segmentos["next_stop_id"] = segmentos["next_stop_id"].astype(stops["stop_id"].dtype)

segmentos["tiempo_segmento_min"] = (
    segmentos["next_arrival_sec"] - segmentos["departure_sec"]
) / 60

segmentos = segmentos[
    segmentos["tiempo_segmento_min"].between(0, 180, inclusive="both")
].copy()

segmentos_medianos = (
    segmentos.groupby(
        ["stop_id", "next_stop_id", "route_id"], as_index=False
    )["tiempo_segmento_min"]
    .median()
)

coords = stops[["stop_lat", "stop_lon"]].to_numpy()
distancia_cuadrada = (
    (coords[:, 0] - CORREGIDORA_LAT) ** 2
    + (coords[:, 1] - CORREGIDORA_LON) ** 2
)
corregidora_idx = int(np.argmin(distancia_cuadrada))
corregidora_stop_id = stops.iloc[corregidora_idx]["stop_id"]
corregidora_stop_name = stops.iloc[corregidora_idx]["stop_name"]

print(
    "Parada GTFS más cercana a Corregidora:",
    corregidora_stop_id,
    "-",
    corregidora_stop_name,
)


Parada GTFS más cercana a Corregidora: 3025 - Estío/Calle Dr. Manuel Domínguez


In [4]:
resultados_base = calcular_rutas_hacia_destino(
    segmentos=segmentos_medianos,
    stops=stops,
    routes=routes,
    destino_stop_id=corregidora_stop_id,
    penalizacion_transbordo_min=PENALIZACION_TRANSBORDO_MIN,
)

RUTAS_BASE_PATH = PROCESSED_DIR / "rutas_base_gtfs.csv"
guardar_rutas_base(resultados_base, RUTAS_BASE_PATH)

# Formato compatible con el cálculo histórico de las correcciones de Google.
df_base = resultados_base.rename(
    columns={"tiempo_red_min": "tiempo_gtfs_min"}
).copy()
df_base["camino_stop_ids"] = df_base["camino_stop_ids"].apply(
    lambda valores: "|".join(map(str, valores))
)
df_base["rutas_camino"] = df_base["rutas_por_segmento"].apply(
    lambda valores: "|".join(map(str, compactar_rutas(valores)))
)
route_name_map = routes.set_index("route_id")["route_short_name"].to_dict()

print(f"{len(df_base):,} paradas tienen camino programado hacia Corregidora.")
print(f"Rutas base exportadas para RutasQroBus: {RUTAS_BASE_PATH}")
display(df_base.head(10))


2,259 paradas tienen camino programado hacia Corregidora.
Rutas base exportadas para RutasQroBus: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/rutas_base_gtfs.csv


,stop_id,tiempo_gtfs_min,route_id_principal,route_short_name,num_transbordos,num_paradas_camino,num_rutas_camino,tipo_conexion,itinerario_route_ids,itinerario_rutas,paradas_transbordo,camino_stop_ids,rutas_por_segmento,penalizacion_transbordo_config_min,stop_name,stop_lat,stop_lon,rutas_camino
0,3025,0.000000,NaN,None,0,1,0,Destino,,,,3025,[],5.0,Estío/Calle Dr. Manuel Domínguez,20.599573,-100.400492,
1,2341,1.533333,344.0,C54,0,2,1,Directa,344,C54,,2341|3025,[344],5.0,Av. Felipe Ángeles/Av. San Roque,20.603829,-100.399494,344
2,3295,2.216667,344.0,C54,0,3,1,Directa,344,C54,,3295|2341|3025,"[344, 344]",5.0,Felipe Ángeles/Fraternidad,20.605753,-100.399474,344
3,2340,2.733333,344.0,C54,0,4,1,Directa,344,C54,,2340|3295|2341|3025,"[344, 344, 344]",5.0,Av. Felipe Ángeles/Plan de Ayala Poniente,20.607204,-100.399456,344
4,3294,3.583333,344.0,C54,0,5,1,Directa,344,C54,,3294|2340|3295|2341|3025,"[344, 344, 344, 344]",5.0,Av. Felipe Ángeles/Calle del Porvenir,20.609586,-100.399427,344
5,2339,4.200000,344.0,C54,0,6,1,Directa,344,C54,,2339|3294|2340|3295|2341|3025,"[344, 344, 344, 344, 344]",5.0,Av. Felipe Ángeles/Felipe Ángeles 225,20.611300,-100.399656,344
6,2338,4.700000,344.0,C54,0,7,1,Directa,344,C54,,2338|2339|3294|2340|3295|2341|3025,"[344, 344, 344, 344, 344, 344]",5.0,Av. Felipe Ángeles/Estadística,20.612684,-100.399821,344
7,4256,5.366667,344.0,C54,0,8,1,Directa,344,C54,,4256|2338|2339|3294|2340|3295|2341|3025,"[344, 344, 344, 344, 344, 344, 344]",5.0,Epigmenio González/Departamental Parques,20.612535,-100.401796,344
8,3363,8.033333,344.0,C54,0,9,1,Directa,344,C54,,3363|4256|2338|2339|3294|2340|3295|2341|3025,"[344, 344, 344, 344, 344, 344, 344, 344]",5.0,Prol. Tecnológico/Calle San Joaquín,20.612622,-100.409849,344
9,3362,9.200000,344.0,C54,0,10,1,Directa,344,C54,,3362|3363|4256|2338|2339|3294|2340|3295|2341|3025,"[344, 344, 344, 344, 344, 344, 344, 344, 344]",5.0,Prol. Tecnológico/Ford Citelis Querétaro,20.615897,-100.410138,344


## 5. Distancias y trazos sobre la red vial

El GTFS local no incluye `shapes.txt`, así que la distancia entre dos paradas consecutivas no puede leerse del feed. En lugar de la línea recta, cada par se enruta sobre la red vial de OpenStreetMap respetando los sentidos de circulación. Las paradas se enganchan a la calle más cercana y el enrutamiento prefiere vialidades principales sobre calles locales.

La red se descarga una vez y se conserva en `data/cache/`; `ACTUALIZAR_RED_OSM=true` fuerza una descarga nueva. Cuando un par no tiene camino, o el camino encontrado es un rodeo inverosímil, se conserva la línea recta y queda marcado en `metodo_trazo`. Los tiempos de autobús siguen viniendo del horario GTFS: este paso corrige distancias y geometrías, no duraciones. Datos cartográficos © [colaboradores de OpenStreetMap](https://www.openstreetmap.org/copyright), ODbL.


In [5]:
RED_VIAL_PATH = CACHE_DIR / "osm_red_vial_qrobus.graphml"
TRAZOS_PATH = PROCESSED_DIR / "segmentos_red_vial.csv"
MARGEN_RED_GRADOS = 0.01

pares_segmentos = segmentos_medianos[["stop_id", "next_stop_id"]].drop_duplicates()
paradas_en_uso = stops[
    stops["stop_id"].isin(
        set(pares_segmentos["stop_id"]) | set(pares_segmentos["next_stop_id"])
    )
]
grafo_vial = cargar_red_osm(
    RED_VIAL_PATH,
    bbox=(
        paradas_en_uso["stop_lon"].min() - MARGEN_RED_GRADOS,
        paradas_en_uso["stop_lat"].min() - MARGEN_RED_GRADOS,
        paradas_en_uso["stop_lon"].max() + MARGEN_RED_GRADOS,
        paradas_en_uso["stop_lat"].max() + MARGEN_RED_GRADOS,
    ),
    network_type="drive",
    actualizar=ACTUALIZAR_RED_OSM,
)
red_vial = RedVial(grafo_vial, paradas_en_uso, factores_via=FACTORES_VIA_AUTOBUS)
trazos_autobus = red_vial.trazar_pares(pares_segmentos)

con_trazo_vial = trazos_autobus["metodo_trazo"].eq(TRAZO_RED_VIAL)
print(
    f"Red vial: {grafo_vial.number_of_nodes():,} nodos y "
    f"{grafo_vial.number_of_edges():,} aristas."
)
print(
    f"Pares de paradas trazados por calles: {con_trazo_vial.sum():,} de "
    f"{len(trazos_autobus):,}; en línea recta: {(~con_trazo_vial).sum():,}."
)
print(
    "Distancia total por calles: "
    f"{trazos_autobus['distancia_red_m'].sum() / 1000:,.0f} km frente a "
    f"{trazos_autobus['distancia_recta_m'].sum() / 1000:,.0f} km en línea recta "
    f"(desvío mediano {trazos_autobus['factor_desvio'].median():.2f})."
)
display(
    trazos_autobus.drop(columns="coordenadas")
    .sort_values("factor_desvio", ascending=False)
    .head(10)
)


Red vial: 70,324 nodos y 162,910 aristas.
Pares de paradas trazados por calles: 3,170 de 3,194; en línea recta: 24.
Distancia total por calles: 1,730 km frente a 1,531 km en línea recta (desvío mediano 1.01).


,stop_id,next_stop_id,distancia_recta_m,distancia_red_m,factor_desvio,metodo_trazo
433,2004,2642,94.2,369.8,3.927,red_vial
2006,3316,3394,142.0,496.1,3.493,red_vial
2193,3488,2572,94.6,324.6,3.430,red_vial
2329,3644,3648,84.2,255.8,3.037,red_vial
2482,3797,3798,209.5,631.4,3.013,red_vial
2272,3575,2520,345.8,977.6,2.827,red_vial
482,2039,2246,622.8,1717.4,2.758,red_vial
2096,3393,3582,139.8,378.8,2.710,red_vial
247,1767,1434,542.8,1457.8,2.686,red_vial
912,2397,5083,326.9,841.5,2.574,red_vial


## 6. Escenario al sur de las vías, sin cruces ferroviarios

Este segundo escenario usa `2312 — Av. Tecnológico/Calle Mariano Escobedo` únicamente como referencia para seleccionar el lado sur de la vía. Antes de calcular los caminos se eliminan todos los segmentos que tocan o cruzan la línea ferroviaria. Las paradas del mismo lado situadas a diez minutos caminando o menos de la estación se conectan a un destino peatonal virtual. Esa caminata se mide sobre la red peatonal de OpenStreetMap, recortada al lado sur para que ningún recorrido cruce la vía, y ya no en línea recta; una parada sin camino peatonal hasta la estación no recibe acceso a pie. Como la estación todavía no tiene accesos publicados, se supone que se entra por la calle del lado sur más cercana a sus coordenadas. Después se ejecuta Dijkstra sobre ese grafo reducido. Esto permite combinar autobús y último tramo a pie sin exigir una ruta que cruce las vías.

La polilínea simplificada corresponde a la `Línea Juárez` de OpenStreetMap en el entorno cubierto por los caminos de 30 minutos. Es una barrera analítica, no un levantamiento de ingeniería. Datos cartográficos © [colaboradores de OpenStreetMap](https://www.openstreetmap.org/copyright), disponibles bajo ODbL.


In [6]:
REFERENCIA_LADO_SUR_STOP_ID = "2312"
DESTINO_PEATONAL_SUR_ID = "ESTACION_CORREGIDORA_SUR"
RUTA_CAMINATA_ID = "CAMINATA"
MARGEN_SEGURIDAD_VIA_M = 15.0
VELOCIDAD_CAMINATA_M_MIN = 80.0
MAX_CAMINATA_FINAL_MIN = 10.0
RADIO_RED_PEATONAL_M = 2_000
RED_PEATONAL_PATH = CACHE_DIR / "osm_red_peatonal_corregidora.graphml"

segmentos_bus_sur = filtrar_segmentos_mismo_lado_vias(
    segmentos=segmentos_medianos,
    stops=stops,
    via_latlon=VIA_FERREA_CORREGIDORA_LATLON,
    referencia_stop_id=REFERENCIA_LADO_SUR_STOP_ID,
    margen_m=MARGEN_SEGURIDAD_VIA_M,
)

referencia_sur = stops[
    stops["stop_id"].astype(str).eq(REFERENCIA_LADO_SUR_STOP_ID)
].iloc[0]
referencia_sur_latlon = (referencia_sur["stop_lat"], referencia_sur["stop_lon"])
stops_sur = stops[
    distancia_firmada_a_via(
        stops["stop_lat"],
        stops["stop_lon"],
        VIA_FERREA_CORREGIDORA_LATLON,
        referencia_sur_latlon,
    )
    >= MARGEN_SEGURIDAD_VIA_M
].copy()

distancia_maxima_caminata_m = MAX_CAMINATA_FINAL_MIN * VELOCIDAD_CAMINATA_M_MIN
stops_sur["distancia_recta_estacion_m"] = haversine_m(
    stops_sur["stop_lat"].to_numpy(),
    stops_sur["stop_lon"].to_numpy(),
    CORREGIDORA_LAT,
    CORREGIDORA_LON,
)
# La línea recta solo descarta paradas: caminar por calles nunca es más corto.
candidatas_caminata = stops_sur[
    stops_sur["distancia_recta_estacion_m"] <= distancia_maxima_caminata_m
].copy()

grafo_peatonal = cargar_red_osm(
    RED_PEATONAL_PATH,
    centro=(CORREGIDORA_LAT, CORREGIDORA_LON),
    radio_m=RADIO_RED_PEATONAL_M,
    network_type="walk",
    actualizar=ACTUALIZAR_RED_OSM,
)
grafo_peatonal_sur = recortar_red_a_lado_de_via(
    grafo_peatonal,
    VIA_FERREA_CORREGIDORA_LATLON,
    referencia_sur_latlon,
    margen_m=MARGEN_SEGURIDAD_VIA_M,
)
puntos_caminata = pd.concat(
    [
        candidatas_caminata[["stop_id", "stop_lat", "stop_lon"]],
        pd.DataFrame(
            [
                {
                    "stop_id": DESTINO_PEATONAL_SUR_ID,
                    "stop_lat": CORREGIDORA_LAT,
                    "stop_lon": CORREGIDORA_LON,
                }
            ]
        ),
    ],
    ignore_index=True,
)
red_peatonal = RedVial(
    grafo_peatonal_sur,
    puntos_caminata,
    tolerancia_candidatos_m=5.0,
    contar_enganche=True,
)
trazos_caminata = red_peatonal.trazar_pares(
    candidatas_caminata.assign(next_stop_id=DESTINO_PEATONAL_SUR_ID),
    max_factor_desvio=None,
)
trazos_caminata = trazos_caminata[
    trazos_caminata["metodo_trazo"].eq(TRAZO_RED_VIAL)
].copy()
accesos_peatonales = trazos_caminata[
    trazos_caminata["distancia_red_m"] <= distancia_maxima_caminata_m
].copy()
accesos_peatonales["stop_id"] = accesos_peatonales["stop_id"].astype(
    stops["stop_id"].dtype
)
segmentos_caminata = pd.DataFrame(
    {
        "stop_id": accesos_peatonales["stop_id"],
        "next_stop_id": DESTINO_PEATONAL_SUR_ID,
        "route_id": RUTA_CAMINATA_ID,
        "tiempo_segmento_min": (
            accesos_peatonales["distancia_red_m"] / VELOCIDAD_CAMINATA_M_MIN
        ),
    }
)
guardar_trazos(
    pd.concat([trazos_autobus, accesos_peatonales], ignore_index=True),
    TRAZOS_PATH,
)
stops_sur_ext = pd.concat(
    [
        stops,
        pd.DataFrame(
            [
                {
                    "stop_id": DESTINO_PEATONAL_SUR_ID,
                    "stop_name": "Estación Corregidora (acceso sur)",
                    "stop_lat": CORREGIDORA_LAT,
                    "stop_lon": CORREGIDORA_LON,
                }
            ]
        ),
    ],
    ignore_index=True,
)
segmentos_sur_vias = pd.concat(
    [segmentos_bus_sur, segmentos_caminata], ignore_index=True
)
routes_sur_ext = pd.concat(
    [
        routes,
        pd.DataFrame(
            [
                {
                    "route_id": RUTA_CAMINATA_ID,
                    "route_short_name": "Caminata",
                    "route_long_name": "Último tramo a pie",
                }
            ]
        ),
    ],
    ignore_index=True,
)

resultados_sur_vias = calcular_rutas_hacia_destino(
    segmentos=segmentos_sur_vias,
    stops=stops_sur_ext,
    routes=routes_sur_ext,
    destino_stop_id=DESTINO_PEATONAL_SUR_ID,
    penalizacion_transbordo_min=PENALIZACION_TRANSBORDO_MIN,
    route_ids_sin_penalizacion={RUTA_CAMINATA_ID},
    route_ids_auxiliares={RUTA_CAMINATA_ID},
)

RUTAS_SUR_VIAS_PATH = PROCESSED_DIR / "rutas_base_sur_vias.csv"
guardar_rutas_base(resultados_sur_vias, RUTAS_SUR_VIAS_PATH)
if REFERENCIA_LADO_SUR_STOP_ID not in set(
    resultados_sur_vias["stop_id"].astype(str)
):
    raise RuntimeError("La parada de referencia no puede llegar sin cruzar la vía.")

print(f"Segmentos generales: {len(segmentos_medianos):,}")
print(f"Segmentos de autobús conservados al sur: {len(segmentos_bus_sur):,}")
print(f"Paradas a {distancia_maxima_caminata_m:g} m o menos en línea recta: {len(candidatas_caminata):,}")
print(f"Con camino peatonal sin cruzar la vía: {len(trazos_caminata):,}")
print(f"Accesos peatonales a {MAX_CAMINATA_FINAL_MIN:g} min o menos por calles: {len(segmentos_caminata):,}")
print(f"Trazos exportados: {TRAZOS_PATH}")
print(f"Paradas alcanzables sin cruzar las vías: {len(resultados_sur_vias):,}")
print(f"Escenario alternativo exportado: {RUTAS_SUR_VIAS_PATH}")


Segmentos generales: 7,751
Segmentos de autobús conservados al sur: 3,535
Paradas a 800 m o menos en línea recta: 20
Con camino peatonal sin cruzar la vía: 20
Accesos peatonales a 10 min o menos por calles: 10
Trazos exportados: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/segmentos_red_vial.csv
Paradas alcanzables sin cruzar las vías: 909
Escenario alternativo exportado: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/rutas_base_sur_vias.csv


## 7. Consulta de Google Maps Routes API

Se utiliza `computeRouteMatrix` con una sola estación destino y hasta 100 paradas origen por lote. La población consultada es la unión, sin duplicados, de las paradas candidatas del escenario general y del escenario al sur de las vías cuyo tiempo de red es menor o igual a `UMBRAL_CONSULTA_GOOGLE_MIN` (por defecto igual a `UMBRAL_ANALISIS_MIN`; 60 minutos para alimentar el escenario de una hora). Una parada nueva solo entra a la clasificación P50/P80 cuando acumula `MIN_OBSERVACIONES_HISTORICAS` consultas. Así se consulta una sola vez cada parada y el escenario alternativo también obtiene una ETA propia.

El modo `TRANSIT` se restringe preferentemente a autobús. La respuesta solo solicita los campos necesarios para controlar costo, tamaño y trazabilidad.

Cada ejecución genera una instantánea con sello UTC. Las observaciones se acumulan temporalmente en `data/processed/google_maps_transit_observaciones.csv`; así el promedio mejora al ejecutar el notebook en distintos días y horarios. El notebook elimina automáticamente filas mayores a `RETENCION_DIAS`. Esta retención es una salvaguarda técnica, no sustituye la revisión del acuerdo vigente de Google Maps aplicable al proyecto.

Las funciones de consulta viven en `scripts/google_routes.py`; este notebook solo decide qué paradas consultar y guarda las respuestas. La consulta incluye reintentos exponenciales para errores temporales (`429` y `5xx`). Los errores permanentes se conservan en la salida en vez de ocultarse.


In [7]:
ids_consulta_general = set(
    df_base.loc[
        df_base["stop_id"].ne(corregidora_stop_id)
        & df_base["tiempo_gtfs_min"].le(UMBRAL_CONSULTA_GOOGLE_MIN),
        "stop_id",
    ]
)
ids_consulta_sur = set(
    resultados_sur_vias.loc[
        resultados_sur_vias["stop_id"].ne(DESTINO_PEATONAL_SUR_ID)
        & resultados_sur_vias["tiempo_red_min"].le(UMBRAL_CONSULTA_GOOGLE_MIN),
        "stop_id",
    ]
)
ids_consulta = ids_consulta_general | ids_consulta_sur
df_consulta = df_base[df_base["stop_id"].isin(ids_consulta)].copy()
if MAX_PARADAS is not None:
    df_consulta = df_consulta.head(int(MAX_PARADAS)).copy()

print(
    f"Candidatas del escenario general: {len(ids_consulta_general):,}; "
    f"del escenario sin cruzar vías: {len(ids_consulta_sur):,}; "
    f"unión sin duplicados: {len(df_consulta):,}. "
    f"Elementos facturables estimados para esta ejecución: {len(df_consulta):,}."
)

if EJECUTAR_GOOGLE_MAPS:
    df_nuevas = consultar_paradas_google(
        df_consulta,
        GOOGLE_MAPS_API_KEY,
        (CORREGIDORA_LAT, CORREGIDORA_LON),
    )
else:
    df_nuevas = pd.DataFrame()
    print(
        "Consulta omitida. Configure GOOGLE_MAPS_API_KEY y "
        "EJECUTAR_GOOGLE_MAPS=true para obtener una instantánea."
    )


Candidatas del escenario general: 1,628; del escenario sin cruzar vías: 756; unión sin duplicados: 1,706. Elementos facturables estimados para esta ejecución: 1,706.
Consulta omitida. Configure GOOGLE_MAPS_API_KEY y EJECUTAR_GOOGLE_MAPS=true para obtener una instantánea.


## 8. Persistencia y control de calidad

Solo se consideran válidos los elementos con `condition="ROUTE_EXISTS"`, sin código de error y con duración numérica. El histórico conserva también los fallos para poder auditar cobertura y disponibilidad.


In [8]:
OBSERVACIONES_PATH = PROCESSED_DIR / "google_maps_transit_observaciones.csv"

if OBSERVACIONES_PATH.exists():
    historico_previo = pd.read_csv(OBSERVACIONES_PATH)
    if "observed_at_utc" in historico_previo.columns:
        fechas_observadas = pd.to_datetime(
            historico_previo["observed_at_utc"],
            errors="coerce", utc=True, format="mixed",
        )
        fechas_salida = pd.to_datetime(
            historico_previo.get("departure_time_utc"),
            errors="coerce", utc=True, format="mixed",
        )
        fechas = fechas_observadas.fillna(fechas_salida)
        historico_previo["observed_at_utc"] = fechas
        limite = pd.Timestamp.now(tz="UTC") - pd.Timedelta(
            RETENCION_HISTORICO_DIAS, unit="D"
        )
        historico_previo = historico_previo[fechas.ge(limite)].copy()
else:
    historico_previo = pd.DataFrame()

df_observaciones = (
    pd.concat([historico_previo, df_nuevas], ignore_index=True)
    if not df_nuevas.empty
    else historico_previo.copy()
)

if not df_observaciones.empty:
    df_observaciones["stop_id"] = df_observaciones["stop_id"].astype(str)
    df_observaciones["observed_at_utc"] = pd.to_datetime(
        df_observaciones["observed_at_utc"],
        errors="coerce",
        utc=True,
        format="mixed",
    )
    # Recupera observaciones antiguas afectadas por formatos ISO mezclados.
    departure_recuperada = pd.to_datetime(
        df_observaciones["departure_time_utc"],
        errors="coerce",
        utc=True,
        format="mixed",
    )
    df_observaciones["observed_at_utc"] = df_observaciones[
        "observed_at_utc"
    ].fillna(departure_recuperada)
    df_observaciones = df_observaciones.drop_duplicates(
        subset=["observed_at_utc", "stop_id"], keep="last"
    )

    # Recalcula la comparación histórica contra la red GTFS vigente.
    # Google ETA se conserva; solo se actualizan tiempo y ruta base.
    base_actual = df_base[
        ["stop_id", "tiempo_gtfs_min", "route_id_principal"]
    ].copy()
    base_actual["stop_id"] = base_actual["stop_id"].astype(str)
    df_observaciones = (
        df_observaciones.drop(
            columns=["tiempo_gtfs_min", "route_id_principal"],
            errors="ignore",
        )
        .merge(base_actual, on="stop_id", how="left", validate="many_to_one")
    )
    df_observaciones["observed_at_local"] = df_observaciones[
        "observed_at_utc"
    ].dt.tz_convert("America/Mexico_City")
    df_observaciones["tipo_dia"] = np.where(
        df_observaciones["observed_at_local"].dt.dayofweek.lt(5),
        "laboral",
        "fin_de_semana",
    )
    df_observaciones["hora_local"] = df_observaciones[
        "observed_at_local"
    ].dt.hour
    df_observaciones["franja_horaria"] = pd.cut(
        df_observaciones["hora_local"],
        bins=[0, 6, 9, 13, 16, 19, 23, 24],
        labels=["00-06", "06-09", "09-13", "13-16", "16-19", "19-23", "23-24"],
        right=False,
        include_lowest=True,
    ).astype("string")
    df_observaciones.to_csv(OBSERVACIONES_PATH, index=False)
    print(f"Histórico alineado con la red GTFS actual: {OBSERVACIONES_PATH}")

if df_observaciones.empty:
    df_validas = pd.DataFrame()
    print("Todavía no existen observaciones de Google para promediar.")
else:
    columnas_numericas = [
        "tiempo_gtfs_min",
        "google_eta_min",
        "status_code",
    ]
    for columna in columnas_numericas:
        df_observaciones[columna] = pd.to_numeric(
            df_observaciones[columna], errors="coerce"
        )

    df_validas = df_observaciones[
        df_observaciones["condition"].eq("ROUTE_EXISTS")
        & df_observaciones["status_code"].fillna(0).eq(0)
        & df_observaciones["google_eta_min"].between(1, 180, inclusive="both")
        & df_observaciones["observed_at_utc"].notna()
        & df_observaciones["tiempo_gtfs_min"].notna()
    ].copy()

    df_validas["desviacion_eta_min"] = (
        df_validas["google_eta_min"] - df_validas["tiempo_gtfs_min"]
    )
    df_validas["atraso_estimado_min"] = (
        df_validas["desviacion_eta_min"].clip(lower=0)
    )

    paradas_consulta = set(df_consulta["stop_id"].astype(str))
    paradas_historicas_cubiertas = set(df_validas["stop_id"]) & paradas_consulta
    cobertura = len(paradas_historicas_cubiertas) / max(len(paradas_consulta), 1)
    print(f"Observaciones totales: {len(df_observaciones):,}")
    print(f"Observaciones válidas: {len(df_validas):,}")
    print(f"Cobertura de paradas de esta población: {cobertura:.1%}")
    print(f"Datos de rutas: Google Maps ©{datetime.now().year} Google")
    display(
        df_validas[
            [
                "observed_at_utc",
                "stop_id",
                "route_id_principal",
                "tipo_dia",
                "franja_horaria",
                "tiempo_gtfs_min",
                "google_eta_min",
                "desviacion_eta_min",
                "atraso_estimado_min",
            ]
        ].head(10)
    )


Histórico alineado con la red GTFS actual: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/google_maps_transit_observaciones.csv
Observaciones totales: 9,802
Observaciones válidas: 9,797
Cobertura de paradas de esta población: 100.0%
Datos de rutas: Google Maps ©2026 Google


,observed_at_utc,stop_id,route_id_principal,tipo_dia,franja_horaria,tiempo_gtfs_min,google_eta_min,desviacion_eta_min,atraso_estimado_min
0,2026-09-14 16:47:07.157796+00:00,2982,350.0,laboral,09-13,25.850000,17.733333,-8.116667,0.00
1,2026-09-14 16:47:07.157796+00:00,2981,325.0,laboral,09-13,28.383333,14.500000,-13.883333,0.00
2,2026-09-14 16:47:07.157796+00:00,3297,97.0,laboral,09-13,26.825000,16.600000,-10.225000,0.00
3,2026-09-14 16:47:07.157796+00:00,3222,348.0,laboral,09-13,24.850000,14.016667,-10.833333,0.00
4,2026-09-14 16:47:07.157796+00:00,3294,344.0,laboral,09-13,3.583333,13.283333,9.700000,9.70
5,2026-09-14 16:47:07.157796+00:00,3284,324.0,laboral,09-13,12.650000,21.000000,8.350000,8.35
6,2026-09-14 16:47:07.157796+00:00,3571,336.0,laboral,09-13,24.833333,14.466667,-10.366667,0.00
7,2026-09-14 16:47:07.157796+00:00,1920,324.0,laboral,09-13,23.200000,19.500000,-3.700000,0.00
8,2026-09-14 16:47:07.157796+00:00,3295,344.0,laboral,09-13,2.216667,5.466667,3.250000,3.25
9,2026-09-14 16:47:07.157796+00:00,5196,324.0,laboral,09-13,22.683333,23.533333,0.850000,0.85


## 9. Resumen histórico robusto por parada

El resultado principal se calcula por parada usando todas las observaciones válidas del periodo de retención. Se reportan P50 (mediana), P80, tamaño de muestra y proporción de ETAs de 30 minutos o menos. P50, P80 y observaciones son indicadores separados: no se suman.

Una parada es **confiable** cuando tiene al menos `MIN_OBSERVACIONES_HISTORICAS` y su P80 no supera el umbral; es **habitual pero variable** cuando P50 cumple y P80 no. Las filas laboral/fin de semana se conservan únicamente como contexto y futura segmentación; no determinan todavía la clasificación principal. La última ETA también se conserva como referencia actual, sin sustituir al histórico.

Los archivos anteriores de promedio por ruta se siguen exportando por compatibilidad, pero `02_RutasQroBus.ipynb` utiliza el nuevo resumen por parada.


In [9]:
PROMEDIO_RUTAS_PATH = PROCESSED_DIR / "promedio_atrasos_google_por_ruta.csv"
CORRECCIONES_PATH = PROCESSED_DIR / "tiempos_corregidos_google.csv"
ESTADISTICAS_STOP_PATH = PROCESSED_DIR / "estadisticas_historicas_google_por_parada.csv"

if df_validas.empty:
    raise RuntimeError("No existen observaciones válidas para calcular P50 y P80.")

# Resumen general: es el que se usa para clasificar accesibilidad.
estadisticas_stop = (
    df_validas.groupby("stop_id", as_index=False)
    .agg(
        p50_historico_min=("google_eta_min", "median"),
        p80_historico_min=("google_eta_min", lambda x: x.quantile(0.80)),
        observaciones=("google_eta_min", "size"),
        consultas_dentro_umbral=(
            "google_eta_min", lambda x: x.le(UMBRAL_ANALISIS_MIN).sum()
        ),
        primera_observacion_utc=("observed_at_utc", "min"),
        ultima_observacion_utc=("observed_at_utc", "max"),
    )
)
estadisticas_stop["proporcion_dentro_umbral"] = (
    estadisticas_stop["consultas_dentro_umbral"]
    / estadisticas_stop["observaciones"]
)

# Contexto por tipo de día: se exporta, pero todavía no clasifica el mapa.
for tipo_dia in ["laboral", "fin_de_semana"]:
    contexto = (
        df_validas[df_validas["tipo_dia"].eq(tipo_dia)]
        .groupby("stop_id", as_index=False)
        .agg(
            **{
                f"observaciones_{tipo_dia}": ("google_eta_min", "size"),
                f"p50_{tipo_dia}_min": ("google_eta_min", "median"),
                f"p80_{tipo_dia}_min": (
                    "google_eta_min", lambda x: x.quantile(0.80)
                ),
            }
        )
    )
    estadisticas_stop = estadisticas_stop.merge(
        contexto, on="stop_id", how="left", validate="one_to_one"
    )

ultima_eta = (
    df_validas.sort_values("observed_at_utc")
    .drop_duplicates("stop_id", keep="last")
    [["stop_id", "google_eta_min", "observed_at_utc", "tipo_dia", "franja_horaria"]]
    .rename(
        columns={
            "google_eta_min": "ultima_eta_min",
            "observed_at_utc": "ultima_eta_utc",
            "tipo_dia": "ultimo_tipo_dia",
            "franja_horaria": "ultima_franja_horaria",
        }
    )
)
estadisticas_stop = estadisticas_stop.merge(
    ultima_eta, on="stop_id", how="left", validate="one_to_one"
)
estadisticas_stop["calidad_muestra"] = np.select(
    [
        estadisticas_stop["observaciones"].ge(
            max(10, MIN_OBSERVACIONES_HISTORICAS)
        ),
        estadisticas_stop["observaciones"].ge(MIN_OBSERVACIONES_HISTORICAS),
        estadisticas_stop["observaciones"].ge(3),
    ],
    ["Alta", "Suficiente", "Baja"],
    default="Insuficiente",
)
muestra_suficiente = estadisticas_stop["observaciones"].ge(
    MIN_OBSERVACIONES_HISTORICAS
)
estadisticas_stop["clasificacion_acceso"] = np.select(
    [
        ~muestra_suficiente,
        estadisticas_stop["p80_historico_min"].le(UMBRAL_ANALISIS_MIN),
        estadisticas_stop["p50_historico_min"].le(UMBRAL_ANALISIS_MIN),
    ],
    [
        "Datos insuficientes",
        "Acceso confiable",
        "Acceso habitual pero variable",
    ],
    default="Fuera de 30 minutos",
)

# Productos anteriores, conservados para compatibilidad y auditoría.
promedio_por_ruta = (
    df_validas.dropna(subset=["route_id_principal"])
    .groupby("route_id_principal", as_index=False)
    .agg(
        observaciones=("stop_id", "size"),
        paradas_observadas=("stop_id", "nunique"),
        atraso_promedio_ruta_min=("atraso_estimado_min", "mean"),
        desviacion_promedio_ruta_min=("desviacion_eta_min", "mean"),
    )
)
promedio_por_ruta["route_short_name"] = promedio_por_ruta[
    "route_id_principal"
].map(route_name_map)
correcciones = df_base.merge(
    promedio_por_ruta[["route_id_principal", "atraso_promedio_ruta_min"]],
    on="route_id_principal", how="left"
)
correcciones["correccion_disponible"] = correcciones[
    "atraso_promedio_ruta_min"
].notna()
correcciones["atraso_promedio_ruta_min"] = correcciones[
    "atraso_promedio_ruta_min"
].fillna(0)
correcciones["tiempo_corregido_min"] = (
    correcciones["tiempo_gtfs_min"]
    + correcciones["atraso_promedio_ruta_min"]
)

estadisticas_stop.to_csv(ESTADISTICAS_STOP_PATH, index=False)
promedio_por_ruta.to_csv(PROMEDIO_RUTAS_PATH, index=False)
correcciones.to_csv(CORRECCIONES_PATH, index=False)
print(f"Estadísticas robustas exportadas: {ESTADISTICAS_STOP_PATH}")
print(f"Promedios por ruta conservados: {PROMEDIO_RUTAS_PATH}")
display(
    estadisticas_stop.sort_values(
        ["clasificacion_acceso", "p80_historico_min"]
    ).head(15)
)


Estadísticas robustas exportadas: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/estadisticas_historicas_google_por_parada.csv
Promedios por ruta conservados: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/processed/promedio_atrasos_google_por_ruta.csv


,stop_id,p50_historico_min,p80_historico_min,observaciones,consultas_dentro_umbral,primera_observacion_utc,ultima_observacion_utc,proporcion_dentro_umbral,observaciones_laboral,p50_laboral_min,p80_laboral_min,observaciones_fin_de_semana,p50_fin_de_semana_min,p80_fin_de_semana_min,ultima_eta_min,ultima_eta_utc,ultimo_tipo_dia,ultima_franja_horaria,calidad_muestra,clasificacion_acceso
1677,5200,5.000000,5.000000,10,10,2026-09-14 16:47:11.840367+00:00,2026-10-09 01:01:01.926077+00:00,1.0,9,5.000000,5.000000,1.0,5.000000,5.000000,5.000000,2026-10-09 01:01:01.926077+00:00,laboral,19-23,Alta,Acceso confiable
1174,3263,6.650000,6.650000,10,10,2026-09-14 16:47:09.286590+00:00,2026-10-09 01:00:57.584658+00:00,1.0,9,6.650000,6.650000,1.0,6.650000,6.650000,6.650000,2026-10-09 01:00:57.584658+00:00,laboral,19-23,Alta,Acceso confiable
1671,5193,6.916667,6.916667,10,10,2026-09-14 16:47:09.286590+00:00,2026-10-09 01:00:56.536399+00:00,1.0,9,6.916667,6.916667,1.0,6.916667,6.916667,6.916667,2026-10-09 01:00:56.536399+00:00,laboral,19-23,Alta,Acceso confiable
793,2680,7.333333,7.333333,10,10,2026-09-14 16:47:09.286590+00:00,2026-10-09 01:00:57.584658+00:00,1.0,9,7.333333,7.333333,1.0,7.333333,7.333333,7.333333,2026-10-09 01:00:57.584658+00:00,laboral,19-23,Alta,Acceso confiable
467,2316,7.916667,7.916667,10,10,2026-09-14 16:47:12.379539+00:00,2026-10-09 01:01:01.926077+00:00,1.0,9,7.916667,7.916667,1.0,7.916667,7.916667,7.916667,2026-10-09 01:01:01.926077+00:00,laboral,19-23,Alta,Acceso confiable
1672,5194,8.483333,8.483333,10,10,2026-09-14 16:47:09.792714+00:00,2026-10-09 01:01:00.813434+00:00,1.0,9,8.483333,8.483333,1.0,8.483333,8.483333,8.483333,2026-10-09 01:01:00.813434+00:00,laboral,19-23,Alta,Acceso confiable
1608,5070,7.766667,8.580000,10,10,2026-09-14 16:47:09.792714+00:00,2026-10-09 01:00:57.584658+00:00,1.0,9,7.766667,8.180000,1.0,9.700000,9.700000,7.766667,2026-10-09 01:00:57.584658+00:00,laboral,19-23,Alta,Acceso confiable
542,2394,8.900000,8.900000,10,10,2026-09-14 16:47:09.286590+00:00,2026-10-09 01:00:55.263935+00:00,1.0,9,8.900000,8.900000,1.0,8.900000,8.900000,8.900000,2026-10-09 01:00:55.263935+00:00,laboral,19-23,Alta,Acceso confiable
466,2315,8.983333,8.983333,10,10,2026-09-14 16:47:09.286590+00:00,2026-10-09 01:00:55.263935+00:00,1.0,9,8.983333,8.983333,1.0,8.983333,8.983333,8.983333,2026-10-09 01:00:55.263935+00:00,laboral,19-23,Alta,Acceso confiable
493,2342,9.416667,9.416667,10,10,2026-09-14 16:47:09.286590+00:00,2026-10-09 01:00:54.116982+00:00,1.0,9,9.416667,9.416667,1.0,9.416667,9.416667,9.416667,2026-10-09 01:00:54.116982+00:00,laboral,19-23,Alta,Acceso confiable


## 10. Paradas clasificadas con el histórico general

El reporte principal usa P50/P80 por parada. El P80 define acceso confiable, P50 identifica acceso habitual pero variable y el tamaño de muestra evita presentar una sola ETA como tendencia. Las columnas laboral y fin de semana son descriptivas y no filtran este resultado.


In [10]:
metadata_stop = df_base[
    ["stop_id", "stop_name", "route_id_principal", "route_short_name", "tiempo_gtfs_min"]
].copy()
metadata_stop["stop_id"] = metadata_stop["stop_id"].astype(str)
reporte_historico = metadata_stop.merge(
    estadisticas_stop, on="stop_id", how="left", validate="one_to_one"
)
for columna in ["observaciones_laboral", "observaciones_fin_de_semana"]:
    reporte_historico[columna] = reporte_historico[columna].fillna(0).astype(int)

rutas_30_min = reporte_historico[
    reporte_historico["clasificacion_acceso"].eq("Acceso confiable")
].sort_values(["p80_historico_min", "route_short_name", "stop_name"])
variables_30 = reporte_historico[
    reporte_historico["clasificacion_acceso"].eq(
        "Acceso habitual pero variable"
    )
]
insuficientes = reporte_historico[
    reporte_historico["clasificacion_acceso"].eq("Datos insuficientes")
]
print(f"Acceso confiable (P80 <= {UMBRAL_ANALISIS_MIN:g}): {len(rutas_30_min):,}")
print(f"Acceso habitual pero variable: {len(variables_30):,}")
print(f"Datos insuficientes: {len(insuficientes):,}")

columnas_reporte = [
    "stop_id", "stop_name", "route_short_name",
    "p50_historico_min", "p80_historico_min",
    "observaciones", "proporcion_dentro_umbral",
    "observaciones_laboral", "observaciones_fin_de_semana",
    "p50_laboral_min", "p80_laboral_min",
    "p50_fin_de_semana_min", "p80_fin_de_semana_min",
    "ultima_eta_min", "ultimo_tipo_dia", "ultima_franja_horaria",
    "calidad_muestra", "clasificacion_acceso",
]
display(reporte_historico[columnas_reporte].head())


Acceso confiable (P80 <= 30): 327
Acceso habitual pero variable: 36
Datos insuficientes: 663


,stop_id,stop_name,route_short_name,p50_historico_min,p80_historico_min,observaciones,proporcion_dentro_umbral,observaciones_laboral,observaciones_fin_de_semana,p50_laboral_min,p80_laboral_min,p50_fin_de_semana_min,p80_fin_de_semana_min,ultima_eta_min,ultimo_tipo_dia,ultima_franja_horaria,calidad_muestra,clasificacion_acceso
0,3025,Estío/Calle Dr. Manuel Domínguez,None,2.850000,2.850000,2.0,1.0,1,1,2.850000,2.850000,2.850000,2.850000,2.850000,laboral,19-23,Insuficiente,Datos insuficientes
1,2341,Av. Felipe Ángeles/Av. San Roque,C54,4.550000,9.766667,10.0,1.0,9,1,4.550000,9.766667,9.766667,9.766667,4.550000,laboral,19-23,Alta,Acceso confiable
2,3295,Felipe Ángeles/Fraternidad,C54,8.916667,12.366667,10.0,1.0,9,1,5.466667,12.366667,12.366667,12.366667,5.466667,laboral,19-23,Alta,Acceso confiable
3,2340,Av. Felipe Ángeles/Plan de Ayala Poniente,C54,6.200000,11.800000,10.0,1.0,9,1,6.200000,11.800000,14.550000,14.550000,6.200000,laboral,19-23,Alta,Acceso confiable
4,3294,Av. Felipe Ángeles/Calle del Porvenir,C54,10.291667,13.426667,10.0,1.0,9,1,7.300000,13.283333,19.300000,19.300000,7.300000,laboral,19-23,Alta,Acceso confiable


## 11. Recomendación de operación

Para que el promedio represente horas pico y valle, ejecute este notebook de forma programada varias veces al día y conserve el histórico. Como mínimo conviene muestrear mañana, mediodía y tarde durante días hábiles y fines de semana.

Antes de automatizar a gran escala:

- configure una cuota diaria y alertas de presupuesto en Google Cloud;
- revise la cobertura de QroBus en Google Transit;
- no publique la llave ni el archivo de observaciones sin revisar las condiciones de uso de Google Maps;
- si se obtiene un feed GTFS-Realtime oficial, reemplace el proxy por el atraso de `TripUpdates`.
